# Fusion of Inertial and High-Resolution Acoustic Data for Privacy-Preserving Human Activity Recognition

In [ ]:
import tensorflow as tf
random_seed = 42
tf.random.set_seed(random_seed)  # set random seed for tensorflow-cpu
from keras.layers import Normalization
from tensorflow.keras.layers import BatchNormalization,Add, Dense, Dropout, MultiHeadAttention, LayerNormalization, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Model
from tensorflow.keras.initializers import TruncatedNormal
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, Callback
from tensorflow_addons.optimizers import AdamW
from sklearn.model_selection import train_test_split 
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import sys

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupKFold

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import random

random.seed(random_seed)  # set random seed for python
np.random.seed(random_seed)  # set random seed for numpy

os.environ['TF_DETERMINISTIC_OPS'] = '1' 
os.environ['CUDA_VISIBLE_DEVICES'] = '1' # set the GPU device id

In [ ]:
sys.path.append(r'../utils')

#clear the cache
if 'functions' in sys.modules:
    del sys.modules['functions']
if 'data_functions' in sys.modules:
    del sys.modules['data_functions']
if 'Transformer' in sys.modules:
    del sys.modules['Transformer']
if 'fusion_function' in sys.modules:
    del sys.modules['fusion_function']

import functions as funcs
import data_functions as func
import Transformer as trans
import fusion_function as fusion_func

In [ ]:
CLASS_LABELS = np.array(
    [
        "usemicrowave",
        "brushteeth",
        "browsevideo",
        "drinkwater",
        "frying",
        "lying",
        "flushing",
        "downstairs",
        "upstairs",
        "sitting",
        "standing",
        "switchdoor",
        "typing",
        "jumping",
        "running",
        "walking",
        "washhands",
        "writing",
        "switchlight",
        "eating"
    ]
)
act_nums = len(CLASS_LABELS) # The number of activity categories
config = {
    'epochs': 50,
    'num_layers': 3,
    'embed_layer_size': 128,
    'fc_layer_size': 256,
    'num_heads': 6,
    'dropout': 0.1,
    'attention_dropout':  0.1,
    'optimizer': 'adam',
    'amsgrad': True,
    'label_smoothing': 0.1,
    'learning_rate': 1e-3,
    'warmup_steps': 10,
    'batch_size': 64,
    'global_clipnorm': 3.0,
    'weight_decay': 1e-4,
}
early_patience = 5 # Early stopping
win_m=300 # Windows length for IMU
step_m=300
win_s=win_m # Windows length for Audio
step_s=step_m

## Model Definition

In [ ]:
class PositionalEmbedding(Layer):
    def __init__(self, units, dropout_rate, **kwargs):
        super(PositionalEmbedding, self).__init__(**kwargs)
        
        self.units = units
        self.projection = Dense(units, kernel_initializer=TruncatedNormal(stddev=0.02))
        
        self.dropout = Dropout(rate=dropout_rate)
    def build(self, input_shape):
        super(PositionalEmbedding, self).build(input_shape)
        self.position = self.add_weight(
            name="position",
            shape=(1, input_shape[1], self.units),
            initializer=TruncatedNormal(stddev=0.02),
            trainable=True,
        )
    def call(self, inputs, training):
        x = self.projection(inputs)
        x = x + self.position

        return self.dropout(x, training=training)

In [ ]:
class Encoder_test(Layer):
    def __init__(
        self,
        embed_dim,
        mlp_dim,
        num_heads,
        dropout_rate,
        attention_dropout_rate,
        **kwargs
    ):
        super(Encoder_test, self).__init__(**kwargs)
        self.mha = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim,
            dropout=attention_dropout_rate,
            kernel_initializer=TruncatedNormal(stddev=0.02),
        )

        self.dense_0 = Dense(
            units=mlp_dim,
            activation="gelu",
            kernel_initializer=TruncatedNormal(stddev=0.02),
        )
        self.dense_1 = Dense(
            units=embed_dim, kernel_initializer=TruncatedNormal(stddev=0.02)
        )

        self.dropout_0 = Dropout(rate=dropout_rate)
        self.dropout_1 = Dropout(rate=dropout_rate)

        self.norm_0 = LayerNormalization(epsilon=1e-5)
        self.norm_1 = LayerNormalization(epsilon=1e-5)

        self.add_0 = Add()
        self.add_1 = Add()

    def call(self, inputs, training):
        # Attention block
        x = self.norm_0(inputs)
       
        x, attn = self.mha(
            query=x,
            value=x,
            key=x,
            return_attention_scores=True,
            training=training,
        )
        x = self.dropout_0(x, training=training)
        x = self.add_0([x, inputs])

        # MLP block
        y = self.norm_1(x)
        y = self.dense_0(y)
        y = self.dense_1(y)
        y = self.dropout_1(y, training=training)

        return self.add_1([x, y]), attn

In [ ]:
class Transformer_test(Layer):
    def __init__(
        self,
        num_layers,
        embed_dim,
        mlp_dim,
        num_heads,
        dropout_rate,
        attention_dropout_rate,
        **kwargs
    ):
        super(Transformer_test, self).__init__(**kwargs)

        # Input
        self.pos_embs = PositionalEmbedding(embed_dim, dropout_rate)

        # Encoder
        self.e_layers = [
            Encoder_test(embed_dim, mlp_dim, num_heads, dropout_rate, attention_dropout_rate)
            for _ in range(num_layers)
        ]


    def call(self, inputs, training):
        x = self.pos_embs(inputs, training=training)
        for layer in self.e_layers:
            x, attn = layer(x, training=training)
        return x, attn

In [ ]:
class CrossAttention(Layer):
    def __init__(self, embed_dim, **kwargs):
        super(CrossAttention, self).__init__(**kwargs)
        
        self.dense_1 = Dense(
            units=embed_dim, kernel_initializer=TruncatedNormal(stddev=0.02)
        )
        self.dense_2 = Dense(
            units=embed_dim, kernel_initializer=TruncatedNormal(stddev=0.02)
        )
        self.dense_3 = Dense(
            units=embed_dim, kernel_initializer=TruncatedNormal(stddev=0.02)
        )
        self.scale = (embed_dim/6)**-0.5
    def call(self, q_input, k_input):
        # calculate attention score
        attention_scores = tf.matmul(q_input, k_input, transpose_b=True)* self.scale
        attention_weights = tf.nn.softmax(attention_scores, axis=-1)
        # obtain the final output through weighting
        weighted_values = tf.matmul(attention_weights, k_input)
        weighted_values += k_input
        return weighted_values

In [ ]:
class SelfAttention(Layer):
    def __init__(self, hidden_size, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)
        self.hidden_size = hidden_size

    def build(self, input_shape):
        self.weight = self.add_weight(shape=(self.hidden_size, 1), initializer='glorot_uniform', trainable=True, name='weight')
        self.bias = self.add_weight(shape=(1,), initializer='zeros', trainable=True, name='bias')

    def call(self, inputs):
        scores = tf.squeeze(tf.matmul(inputs, self.weight), axis=2) + self.bias
        attn_weights = tf.expand_dims(tf.nn.softmax(scores), axis=2)
        attn_output = tf.reduce_sum(attn_weights * inputs, axis=1)

        return attn_output

In [ ]:
class Multi_Transformer_hybridfusion(Model):
    def __init__(
        self,
        num_layers,
        embed_dim,
        mlp_dim,
        num_heads,
        num_classes,
        dropout_rate,
        attention_dropout_rate,
        **kwargs
    ):
        super(Multi_Transformer_hybridfusion, self).__init__(**kwargs)
        self.input_norm1 = Normalization()# normalize the input
        self.input_norm2 = Normalization()# normalize the input

        self.motion_net = Transformer_test(num_layers,embed_dim,mlp_dim,num_heads,dropout_rate,attention_dropout_rate, name='motion_net')
        self.sound_net = Transformer_test(num_layers,embed_dim,mlp_dim,num_heads,dropout_rate,attention_dropout_rate,name='sound_net')
        self.catt = CrossAttention(embed_dim)
        self.satt = SelfAttention(2*config['embed_layer_size'])
        self.satt2 = SelfAttention(win_m)
        self.norm = LayerNormalization(epsilon=1e-5)
        self.final_layer = Dense(num_classes, kernel_initializer="zeros")

    def call(self, inputs, training):
        motion, sound = inputs
        m_x = self.input_norm1(motion)
        s_x = self.input_norm2(sound)
        m, m_attn = self.motion_net(m_x, training=training)
        s, s_attn = self.sound_net(s_x, training=training)
        
        concat_feature = tf.concat([m, s], axis=-1)
        concat_feature_tp = tf.transpose(concat_feature, perm=[0, 2, 1])
        
        sa_1 = concat_feature + tf.expand_dims(self.satt2(concat_feature_tp), axis=-1)
        sa_2 = tf.transpose(concat_feature_tp + tf.expand_dims(self.satt(concat_feature), axis=-1), perm=[0, 2, 1])

        ca_1 = self.catt(m, s)
        ca_2 = self.catt(s, m)

        final = tf.concat([sa_1 + sa_2, ca_1, ca_2], axis=-1)
        normlayer = self.norm(final)
        output = self.final_layer(normlayer)
        return output
    def get_config(self):
        config = super(Multi_Transformer_hybridfusion, self).get_config()
        return config
    @classmethod
    def from_config(cls, config):
        # recreate the model instance according to the configuration information
        return cls()

In [ ]:
def smoothed_sparse_categorical_crossentropy(num_classes,label_smoothing: float = 0.0):
    def loss_fn(y_true, y_pred):
        y_true = tf.one_hot(y_true, num_classes)
        loss = tf.keras.losses.categorical_crossentropy(y_true, y_pred, from_logits=True, label_smoothing=label_smoothing)
        return tf.reduce_mean(loss)
    return loss_fn
    
"""LR Sheduler"""
def cosine_schedule(base_lr, total_steps, warmup_steps):
    def step_fn(epoch):
        lr = base_lr
        epoch += 1 
        progress = (epoch - warmup_steps) / float(total_steps - warmup_steps)
        progress = tf.clip_by_value(progress, 0.0, 1.0)
        
        lr = lr * 0.5 * (1.0 + tf.cos(math.pi * progress))

        if warmup_steps:
            lr = lr * tf.minimum(1.0, epoch / warmup_steps)
        return lr
    return step_fn


In [ ]:
def train(m_train_X, s_train_X, train_y, m_val_x, s_val_x, val_y, config, weights_path):
    # Generate new model
    model = Multi_Transformer_hybridfusion(
        num_layers=config['num_layers'],
        embed_dim=config['embed_layer_size'],  
        mlp_dim=config['fc_layer_size'],
        num_heads=config['num_heads'],
        num_classes=act_nums,
        dropout_rate=config['dropout'],
        attention_dropout_rate=config['attention_dropout']
    )
    # adapt on training dataset - must be before model.compile !
    model.input_norm1.adapt(m_train_X, batch_size=config['batch_size'])
    model.input_norm2.adapt(s_train_X, batch_size=config['batch_size'])

    # Select optimizer
    if config['optimizer'] == "adam":
        optim = Adam(
            global_clipnorm=config['global_clipnorm'],
            amsgrad=config['amsgrad'],
        )
    elif config['optimizer'] == "adamw":
        optim = AdamW(
            weight_decay=config['weight_decay'],
            amsgrad=config['amsgrad'],
            global_clipnorm=config['global_clipnorm'],
            exclude_from_weight_decay=["position"],
        )
    else:
        raise ValueError("The used optimizer is not in list of available")

    model.compile(
        loss=smoothed_sparse_categorical_crossentropy(act_nums,
            label_smoothing=config['label_smoothing']
        ),
        optimizer=optim,
        metrics=["accuracy"],
    )

    reduce_lr_loss = (LearningRateScheduler(cosine_schedule(base_lr=config['learning_rate'],total_steps=config['epochs'],warmup_steps=config['warmup_steps'],)),)
    earlyStopping = EarlyStopping(monitor="val_accuracy", mode="max", min_delta=0.001, patience=early_patience)

    model.fit(
        [m_train_X,s_train_X],
        train_y,
        epochs=config['epochs'],
        batch_size=config['batch_size'],
        validation_data=([m_val_x, s_val_x], val_y),
        callbacks=[reduce_lr_loss, earlyStopping],#cp_callback
        verbose=1,
    )
    model.summary()
    model.save_weights(weights_path)
    return model

In [ ]:
new_motion = r'../../../Processed_data/IMU/processed_IMU.csv'
new_sound = r'../../../Processed_data/Audio/processed_audio_8k-96k.csv'

In [ ]:
"""load dataset"""
print('---------IMU------------')
motion_df = fusion_func.read_csv(new_motion)
motion_signals, motion_labels, motion_subject = fusion_func.load_datasets(motion_df)
print(f'motion_signals.shape:{motion_signals.shape} \n motion_labels.shape:{motion_labels.shape} \n motion_subject.shape:{motion_subject.shape}')

print('---------Sound------------')
sound_df = fusion_func.read_csv(new_sound)
sound_signals, sound_labels, sound_subject = fusion_func.load_datasets(sound_df)
print(f'sound_signals.shape:{sound_signals.shape}\n sound_labels.shape:{sound_labels.shape} \n sound_subject.shape:{sound_subject.shape}')


In [ ]:
"""train、test"""
for i in range(15):
    weights_folder = r"../../finalresult/hybrid_896"
    model_name = "test"+str(i+1)+f".h5"  # set model name
    confusion_matrix_path = r'../../finalresult/confusion_matrix'
    os.makedirs(weights_folder, exist_ok=True)
    os.makedirs(confusion_matrix_path, exist_ok=True)
    weights_path = os.path.join(weights_folder, model_name)
    full_group = {100101,100102,100201,100202,100301,100302,100401,100402,100501,100502,100601,100602,100701,100702,100801,100802,
              100901,100902,101001,101002,101101,101102,101201,101202,101301,101302,101401,101402,101501,101502}
    test_group = {int('10'+str(i+1).zfill(2)+'01'),int('10'+str(i+1).zfill(2)+'02')}
    train_group = full_group - test_group
    print("------------------------------------------------------------------------------------------------------------")
    print("test_group:" + str(test_group))
    m_train_X, m_train_y, m_train_g, m_test_X, m_test_y, m_test_g = fusion_func.pkl_to_npz_new(motion_signals, motion_labels, motion_subject,train_group, test_group, win_m, step_m)
    s_train_X, s_train_y, s_train_g, s_test_X, s_test_y, s_test_g = fusion_func.pkl_to_npz_new(sound_signals, sound_labels, sound_subject,train_group, test_group, win_s, step_s)
    
    """split dataset: split the train set into a validation set further"""
    m_train_X, m_val_X, s_train_X, s_val_X, train_y, val_y = train_test_split(
        m_train_X, s_train_X, m_train_y, test_size=0.15, random_state=random_seed, stratify=m_train_y
        )
    #print(f"train、val、act shape：{m_train_X.shape}, {m_val_X.shape}, {s_train_X.shape}, {s_val_X.shape}, {train_y.shape}, {val_y.shape}")
    
    """Dataset visualization"""
    #figure_new(CLASS_LABELS,motion_labels,train_y,m_test_y,val_y)
    model = train(m_train_X, s_train_X, train_y, m_val_X, s_val_X, val_y, config, weights_path)
    batch_size = 32
    def get_predictions(start, end):
        output = model([m_test_X[start:end], s_test_X[start:end]])
        predictions = np.argmax(output, axis=-1)
        return predictions

    full_predictions = []
    for j in range(m_test_X.shape[0] // batch_size):
        y = get_predictions(j * batch_size, (j + 1) * batch_size)
        full_predictions.append(y)

    y = get_predictions((j + 1) * batch_size, m_test_X.shape[0])
    full_predictions.append(y)

    full_predictions = np.concatenate(full_predictions, axis=0)
    full_predictions = full_predictions.reshape(-1, full_predictions.shape[-1])
    fig, ax = plt.subplots(figsize=(15, 15))
    cm = confusion_matrix(
        CLASS_LABELS[m_test_y.reshape(-1)],
        CLASS_LABELS[full_predictions.reshape(-1)],
        labels=CLASS_LABELS,
        #normalize='true'
    )
    cm1 = confusion_matrix(
        CLASS_LABELS[m_test_y.reshape(-1)],
        CLASS_LABELS[full_predictions.reshape(-1)],
        labels=CLASS_LABELS,
        normalize='true'
    )

    cm_display = ConfusionMatrixDisplay(
        confusion_matrix=cm1, display_labels=CLASS_LABELS
    ).plot(cmap="Blues", xticks_rotation=70, ax=ax)

    report = classification_report(
        CLASS_LABELS[m_test_y.reshape(-1)],
        CLASS_LABELS[full_predictions.reshape(-1)],
        labels=CLASS_LABELS,
        digits=3,
    )
    print(report)
    np.savetxt(os.path.join(confusion_matrix_path, r'cm_hybrid_896_test'+str(i+1)+f'raw.txt'), cm)
    np.savetxt(os.path.join(confusion_matrix_path, r'cm_hybrid_896_test'+str(i+1)+f'nor.txt'), cm1)

    with open(os.path.join(confusion_matrix_path, f'report_hybrid_896_test'+str(i+1)+f'.txt'), "w") as file:
        file.write(report)